# FlytBase Drone Visual Intelligence — Master Hackathon Pipeline

This notebook combines all levels of the computer vision & traffic analytics pipeline:
- **Level 1**: VisDrone YOLO11s Detection, Sliced Inference & ByteTrack Multi-Object Tracking
- **Level 2**: Road Masking, Telemetry Parsing (SRT), GCP Homography & Kinematics (Speeds, Trajectories)
- **Level 3**: Aggregate Traffic Analytics, Turn Movements, Queue Dynamics & Modal Split


## 1. Setup & Dependencies


In [ ]:
!pip install -q ultralytics supervision huggingface_hub pandas pyarrow opencv-python matplotlib scipy
import os, sys, cv2, time, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import supervision as sv
from ultralytics import YOLO
from huggingface_hub import hf_hub_download


## 2. Tracking Pipeline (VisDrone YOLO11s + Tiled Inference + ByteTrack)


In [ ]:
import os
VID = "/content/drive/MyDrive/Visual Intelligence Hackathon/Intersection_Merged.MP4"  # ← paste from Cell 2
assert os.path.exists(VID), "WRONG PATH — fix before continuing"
print("ok", os.path.getsize(VID)/1e9, "GB")

In [ ]:
%%writefile track.py
import argparse, inspect, time
from pathlib import Path
import cv2, numpy as np, pandas as pd
import supervision as sv
from ultralytics import YOLO
from huggingface_hub import hf_hub_download

VISDRONE={0:"pedestrian",1:"people",2:"bicycle",3:"car",4:"van",5:"truck",
          6:"tricycle",7:"awning-tricycle",8:"bus",9:"motor"}
GROUP={"pedestrian":"pedestrian","people":"pedestrian","bicycle":"cyclist",
       "car":"car","van":"LGV","truck":"HGV","bus":"bus","motor":"motorcycle",
       "tricycle":"three-wheeler","awning-tricycle":"three-wheeler"}

def make_tracker(fps):
    sig=set(inspect.signature(sv.ByteTrack.__init__).parameters); kw={}
    if "frame_rate" in sig: kw["frame_rate"]=int(fps)
    for a,b in [("lost_track_buffer",int(fps*4)),("track_buffer",int(fps*4)),
                ("track_activation_threshold",0.20),("track_thresh",0.20),
                ("minimum_matching_threshold",0.85),("match_thresh",0.85)]:
        if a in sig and a not in kw: kw[a]=b
    print("[tracker]",kw,flush=True)
    return sv.ByteTrack(**kw)

_Q={"m":None}
def predict(model,imgs,device,half,tile,conf):
    if _Q["m"] is None:
        try:
            r=model.predict(imgs,device=device,imgsz=tile,conf=conf,verbose=False,
                            quantize=(16 if half else None))
            _Q["m"]="q"; return r
        except TypeError: _Q["m"]="h"
    if _Q["m"]=="q":
        return model.predict(imgs,device=device,imgsz=tile,conf=conf,verbose=False,
                             quantize=(16 if half else None))
    return model.predict(imgs,device=device,imgsz=tile,conf=conf,verbose=False,half=half)

def tiles(h,w,t,ov):
    s=max(int(t*(1-ov)),1)
    xs=sorted(set(list(range(0,max(w-t,1),s))+[max(w-t,0)]))
    ys=sorted(set(list(range(0,max(h-t,1),s))+[max(h-t,0)]))
    return [(x,y) for y in ys for x in xs]

def detect(model,fr,device,half,t,ov,batch,conf,iou):
    h,w=fr.shape[:2]; co=tiles(h,w,t,ov)
    cr=[fr[y:y+t,x:x+t] for x,y in co]; parts=[]
    for i in range(0,len(cr),batch):
        for r,(x,y) in zip(predict(model,cr[i:i+batch],device,half,t,conf),co[i:i+batch]):
            d=sv.Detections.from_ultralytics(r)
            if len(d):
                d.xyxy=d.xyxy+np.array([x,y,x,y],dtype=np.float32); parts.append(d)
    if not parts: return sv.Detections.empty()
    return sv.Detections.merge(parts).with_nms(threshold=iou,class_agnostic=True)

def frames(cap,s,e,st):
    if s: cap.set(cv2.CAP_PROP_POS_FRAMES,s)
    i=s
    while True:
        if e and i>=e: break
        if not cap.grab(): break
        if (i-s)%st==0:
            ok,f=cap.retrieve()
            if ok: yield i,f
        i+=1

def main():
    p=argparse.ArgumentParser()
    p.add_argument("--video",required=True); p.add_argument("--out-dir",default="/content/out")
    p.add_argument("--start-sec",type=float,default=0); p.add_argument("--max-seconds",type=float,default=0)
    p.add_argument("--stride",type=int,default=3); p.add_argument("--tile",type=int,default=1024)
    p.add_argument("--overlap",type=float,default=0.2); p.add_argument("--batch",type=int,default=12)
    p.add_argument("--conf",type=float,default=0.10); p.add_argument("--nms-iou",type=float,default=0.55)
    p.add_argument("--min-track-len",type=int,default=3); p.add_argument("--device",default="cuda")
    a=p.parse_args()
    out=Path(a.out_dir); out.mkdir(parents=True,exist_ok=True)
    half=a.device.startswith("cuda")
    model=YOLO(hf_hub_download("dronefreak/visdrone-yolov11s","best.pt"))
    cap=cv2.VideoCapture(a.video); assert cap.isOpened(), f"cannot open {a.video}"
    fps=cap.get(cv2.CAP_PROP_FPS) or 29.97
    total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    W=int(cap.get(3)); H=int(cap.get(4))
    s=int(a.start_sec*fps); e=total if a.max_seconds<=0 else min(total,s+int(a.max_seconds*fps))
    n_samp=max((e-s)//a.stride,1)
    print(f"{W}x{H} @{fps:.2f} frames {s}->{e} sampled {n_samp}",flush=True)
    tracker=make_tracker(fps/a.stride)
    rows=[]; t0=time.time(); n=0
    for fi,fr in frames(cap,s,e,a.stride):
        det=tracker.update_with_detections(
            detect(model,fr,a.device,half,a.tile,a.overlap,a.batch,a.conf,a.nms_iou))
        for i in range(len(det)):
            if det.tracker_id is None or det.tracker_id[i] is None: continue
            x1,y1,x2,y2=det.xyxy[i]; cid=int(det.class_id[i]); cn=VISDRONE.get(cid,str(cid))
            rows.append(dict(frame_idx=fi,time_sec=fi/fps,track_id=int(det.tracker_id[i]),
                class_id=cid,class_name=cn,class_group=GROUP.get(cn,cn),
                conf=float(det.confidence[i]),x1=float(x1),y1=float(y1),
                x2=float(x2),y2=float(y2),cx=float((x1+x2)/2),cy_foot=float(y2),
                w=float(x2-x1),h=float(y2-y1),interpolated=False))
        n+=1
        if n%50==0:
            r=n/(time.time()-t0)
            print(f"{n}/{n_samp} {r:.2f} f/s ETA {(n_samp-n)/r/60:.1f} min",flush=True)
    cap.release()
    df=pd.DataFrame(rows)
    if df.empty: print("NO DETECTIONS"); return
    k=df.groupby("track_id").size(); df=df[df.track_id.isin(k[k>=a.min_track_len].index)].copy()
    v=(df.groupby(["track_id","class_group"])["conf"].sum().reset_index()
        .sort_values("conf",ascending=False).drop_duplicates("track_id")
        .set_index("track_id")["class_group"])
    df["class_group"]=df.track_id.map(v)
    df.to_parquet(out/"tracks.parquet",index=False)
    per=df.groupby("frame_idx").size(); L=df.groupby("track_id").size()
    print(f"\n=== RAW QC ===\nrows {len(df)} tracks {df.track_id.nunique()}")
    print(f"objects/frame mean {per.mean():.1f}")
    print(f"track len mean {L.mean():.1f} median {L.median():.0f}")
    print(f"wall {(time.time()-t0)/60:.1f} min")

if __name__=="__main__": main()

In [ ]:
%%writefile stitch.py
import sys, numpy as np, pandas as pd
def stitch(df,fps=29.97,max_gap_sec=2.0,gate_mult=3.0,min_gate=60.0):
    mg=int(max_gap_sec*fps)
    real=df[~df.interpolated] if "interpolated" in df else df
    info={}
    for tid,g in real.groupby("track_id"):
        g=g.sort_values("frame_idx"); n=min(4,len(g))
        def vel(s):
            if len(s)<2: return np.array([0.,0.])
            dt=s.frame_idx.iloc[-1]-s.frame_idx.iloc[0]
            if dt==0: return np.array([0.,0.])
            return np.array([(s.cx.iloc[-1]-s.cx.iloc[0])/dt,
                             (s.cy_foot.iloc[-1]-s.cy_foot.iloc[0])/dt])
        info[tid]=dict(f0=int(g.frame_idx.iloc[0]),f1=int(g.frame_idx.iloc[-1]),
            p0=np.array([g.cx.iloc[0],g.cy_foot.iloc[0]]),
            p1=np.array([g.cx.iloc[-1],g.cy_foot.iloc[-1]]),
            v0=vel(g.iloc[:n]),v1=vel(g.iloc[-n:]),
            size=float(np.sqrt(max(g.w.median()*g.h.median(),1))),cls=g.class_group.iloc[0])
    par={t:t for t in info}
    def find(x):
        while par[x]!=x: par[x]=par[par[x]]; x=par[x]
        return x
    byf0=sorted(info,key=lambda t:info[t]["f0"]); used=set(); pr=0
    for a in sorted(info,key=lambda t:info[t]["f1"]):
        A=info[a]; best=None; bd=1e18
        for b in byf0:
            if b==a or b in used: continue
            B=info[b]; gap=B["f0"]-A["f1"]
            if gap<=0: continue
            if gap>mg: break
            if A["cls"]!=B["cls"]: continue
            d=min(np.linalg.norm(A["p1"]+A["v1"]*gap-B["p0"]),
                  np.linalg.norm(B["p0"]-B["v0"]*gap-A["p1"]))
            gate=max(gate_mult*max(A["size"],B["size"]),min_gate)*(1+gap/mg)
            if d<gate and d<bd: bd,best=d,b
        if best is not None: used.add(best); pr+=1; par[find(best)]=find(a)
    df=df.copy(); df["track_id"]=df.track_id.map(lambda t: find(t) if t in par else t)
    return df,pr
def reinterp(df,stride):
    out=[]
    for tid,g in df.groupby("track_id"):
        g=g.sort_values("frame_idx").drop_duplicates("frame_idx").set_index("frame_idx")
        g=g.reindex(range(int(g.index.min()),int(g.index.max())+1,stride))
        g["interpolated"]=g["cx"].isna()
        c=["x1","y1","x2","y2","cx","cy_foot","w","h","conf"]
        g[c]=g[c].interpolate("linear"); g["track_id"]=tid
        for k in ["class_id","class_name","class_group"]: g[k]=g[k].ffill().bfill()
        out.append(g.reset_index().rename(columns={"index":"frame_idx"}))
    return pd.concat(out,ignore_index=True)
if __name__=="__main__":
    src,dst=sys.argv[1],sys.argv[2]; st=int(sys.argv[3]) if len(sys.argv)>3 else 3
    df=pd.read_parquet(src); print(f"before {df.track_id.nunique()} tracks")
    df,pr=stitch(df); print(f"merged {pr} pairs")
    df=reinterp(df,st); df["time_sec"]=df.frame_idx/29.97
    df.to_parquet(dst,index=False)
    L=df.groupby("track_id").size()
    print(f"after {df.track_id.nunique()} tracks")
    print(f"len mean {L.mean():.1f} median {L.median():.0f} p90 {L.quantile(.9):.0f}")
    print(f"objects/frame mean {df.groupby('frame_idx').size().mean():.1f}")
    print(f"interpolated {100*df.interpolated.mean():.1f}%")
    print(df.groupby('track_id').class_group.first().value_counts())

In [ ]:
%%writefile render.py
import sys, cv2, numpy as np, pandas as pd
C={"car":(0,200,255),"LGV":(0,255,120),"HGV":(255,80,0),"bus":(255,0,200),
   "motorcycle":(80,80,255),"cyclist":(255,255,0),"pedestrian":(0,255,255),
   "three-wheeler":(180,120,255)}
pq,vid,out=sys.argv[1],sys.argv[2],sys.argv[3]
t0=float(sys.argv[4]) if len(sys.argv)>4 else 0
dur=float(sys.argv[5]) if len(sys.argv)>5 else 90
df=pd.read_parquet(pq)
st=int(np.median(np.diff(sorted(df.frame_idx.unique())))) or 3
cap=cv2.VideoCapture(vid); assert cap.isOpened()
fps=cap.get(cv2.CAP_PROP_FPS) or 29.97
W=int(cap.get(3)); H=int(cap.get(4)); OW=1920; OH=int(1920*H/W); sx,sy=OW/W,OH/H
f0=max(int(t0*fps),int(df.frame_idx.min())); f1=min(int((t0+dur)*fps),int(df.frame_idx.max()))
d=df[(df.frame_idx>=f0)&(df.frame_idx<=f1)]
by={k:v for k,v in d.groupby("frame_idx")}
vw=cv2.VideoWriter(out,cv2.VideoWriter_fourcc(*"mp4v"),fps/st,(OW,OH))
cap.set(cv2.CAP_PROP_POS_FRAMES,f0); i=f0; n=0
while i<=f1:
    if not cap.grab(): break
    if i in by:
        ok,fr=cap.retrieve()
        if ok:
            fr=cv2.resize(fr,(OW,OH))
            for r in by[i].itertuples():
                c=C.get(r.class_group,(255,255,255))
                p1=(int(r.x1*sx),int(r.y1*sy)); p2=(int(r.x2*sx),int(r.y2*sy))
                cv2.rectangle(fr,p1,p2,c,2)
                cv2.putText(fr,str(int(r.track_id)),(p1[0],max(p1[1]-4,10)),
                            cv2.FONT_HERSHEY_SIMPLEX,0.45,c,1,cv2.LINE_AA)
            cv2.putText(fr,f"t={i/fps:6.1f}s  n={len(by[i])}",(12,28),
                        cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,255),2,cv2.LINE_AA)
            vw.write(fr); n+=1
    i+=1
cap.release(); vw.release()
print(f"wrote {out}: {n} frames, {n*st/fps:.1f}s")

In [ ]:
!python track.py --video "$VID" --out-dir /content/clip90 \
  --start-sec 60 --max-seconds 90 --stride 3 --device cuda

In [ ]:
!python stitch.py /content/clip90/tracks.parquet /content/clip90/tracks_final.parquet 3
!python render.py /content/clip90/tracks_final.parquet "$VID" /content/clip.mp4 60 90
!ffmpeg -y -loglevel error -i /content/clip.mp4 -vcodec libx264 -pix_fmt yuv420p -crf 23 /content/clip_h264.mp4
!mkdir -p /content/drive/MyDrive/flytbase_out
!cp /content/clip90/tracks_final.parquet /content/clip_h264.mp4 /content/drive/MyDrive/flytbase_out/
!ffmpeg -y -loglevel error -i "$VID" -map 0:s:0 /content/drive/MyDrive/flytbase_out/int.srt
!ls -la /content/drive/MyDrive/flytbase_out/

In [ ]:
%matplotlib inline
!pip install -q ipympl

In [ ]:
print(pts)   # paste into POLYS

In [ ]:
from google.colab import files
up = files.upload()   # select the makesense .json

In [ ]:
import cv2
cap=cv2.VideoCapture(VID); cap.set(cv2.CAP_PROP_POS_FRAMES,int(60*29.97))
ok,ref=cap.read(); cap.release(); assert ok
H,W=ref.shape[:2]; print("video frame:",W,H)

In [ ]:
import pandas as pd, numpy as np
mask=np.load('/content/road_mask.npy')
d=pd.read_parquet('/content/clip90/tracks_final.parquet')
xs=d.cx.clip(0,W-1).astype(int).values; ys=d.cy_foot.clip(0,H-1).astype(int).values
d['in_road']=mask[ys,xs]>0
frac=d.groupby('track_id').in_road.mean()
keep=frac[frac>=0.30].index
before=d.track_id.nunique()
d=d[d.track_id.isin(keep)].copy()
d.to_parquet('/content/clip90/tracks_masked.parquet',index=False)
print(f"tracks {before} -> {d.track_id.nunique()} (dropped {before-d.track_id.nunique()})")
print(d.groupby('track_id').class_group.first().value_counts())

In [ ]:
import numpy as np, pandas as pd, cv2
d=pd.read_parquet('/content/clip90/tracks_masked.parquet')
polys=[np.array(p,np.int32) for p in POLYS]
def edge_dist(x,y):
    return min(abs(cv2.pointPolygonTest(p,(float(x),float(y)),True)) for p in polys)
EDGE=140
rec=[]
for tid,g in d.groupby('track_id'):
    g=g.sort_values('frame_idx')
    rec.append(dict(track_id=tid,cls=g.class_group.iloc[0],n=len(g),dur=len(g)*3/29.97,
        d_in=edge_dist(g.cx.iloc[0],g.cy_foot.iloc[0]),
        d_out=edge_dist(g.cx.iloc[-1],g.cy_foot.iloc[-1]),
        disp=float(np.hypot(g.cx.iloc[-1]-g.cx.iloc[0],g.cy_foot.iloc[-1]-g.cy_foot.iloc[0]))))
r=pd.DataFrame(rec)
r['enters']=r.d_in<EDGE; r['exits']=r.d_out<EDGE
r['complete']=r.enters&r.exits; r['stationary']=r.disp<60
r.to_csv('/content/clip90/track_quality.csv',index=False)
mv=r[~r.stationary]
print(f"tracks {len(r)}  moving {len(mv)}  stationary {r.stationary.sum()}")
print(f"COMPLETE: {mv.complete.mean()*100:.1f}%")
print(f"  starts mid-scene: {(~mv.enters).mean()*100:.1f}%")
print(f"  ends mid-scene:   {(~mv.exits).mean()*100:.1f}%")
print(f"median dur {mv.dur.median():.1f}s  p90 {mv.dur.quantile(.9):.1f}s")
print(mv.groupby('cls').complete.agg(['size','mean']))

In [ ]:
%%writefile trace.py
import sys, cv2, numpy as np, pandas as pd
C={"car":(0,200,255),"LGV":(0,255,120),"HGV":(255,80,0),"bus":(255,0,200),
   "motorcycle":(80,80,255),"cyclist":(255,255,0),"pedestrian":(0,255,255),
   "three-wheeler":(180,120,255)}
pq,vid,mk,out=sys.argv[1],sys.argv[2],sys.argv[3],sys.argv[4]
TAIL=int(sys.argv[5]) if len(sys.argv)>5 else 100
df=pd.read_parquet(pq); mask=np.load(mk)
st=int(np.median(np.diff(sorted(df.frame_idx.unique())))) or 3
cap=cv2.VideoCapture(vid); assert cap.isOpened()
fps=cap.get(cv2.CAP_PROP_FPS) or 29.97
W=int(cap.get(3)); H=int(cap.get(4)); OW=1920; OH=int(1920*H/W); sx,sy=OW/W,OH/H
msk=cv2.resize(mask,(OW,OH))
f0,f1=int(df.frame_idx.min()),int(df.frame_idx.max())
by={k:v for k,v in df.groupby("frame_idx")}; hist={}
vw=cv2.VideoWriter(out,cv2.VideoWriter_fourcc(*"mp4v"),fps/st,(OW,OH))
cap.set(cv2.CAP_PROP_POS_FRAMES,f0); i=f0; n=0
while i<=f1:
    if not cap.grab(): break
    if i in by:
        ok,fr=cap.retrieve()
        if ok:
            fr=cv2.resize(fr,(OW,OH))
            fr[msk==0]=(fr[msk==0]*0.35).astype(np.uint8)
            layer=fr.copy()
            for r in by[i].itertuples():
                p=(int(r.cx*sx),int(r.cy_foot*sy))
                hist.setdefault(r.track_id,[]).append(p)
                pts=hist[r.track_id][-TAIL:]; c=C.get(r.class_group,(255,255,255))
                for k in range(1,len(pts)):
                    a=k/len(pts)
                    cv2.line(layer,pts[k-1],pts[k],tuple(int(v*a) for v in c),
                             max(1,int(1+2*a)),cv2.LINE_AA)
                cv2.circle(fr,p,4,c,-1)
                cv2.putText(fr,str(int(r.track_id)),(p[0]+6,p[1]-6),0,0.45,c,1,cv2.LINE_AA)
            fr=cv2.addWeighted(layer,0.85,fr,0.15,0)
            cv2.putText(fr,f"t={i/fps:6.1f}s active={len(by[i])} tracks={len(hist)}",
                        (12,30),0,0.9,(255,255,255),2,cv2.LINE_AA)
            vw.write(fr); n+=1
    i+=1
cap.release(); vw.release(); print(f"wrote {out}: {n} frames")

In [ ]:
!python trace.py /content/clip90/tracks_masked.parquet "$VID" /content/road_mask.npy /content/trace.mp4 100
!ffmpeg -y -loglevel error -i /content/trace.mp4 -vcodec libx264 -pix_fmt yuv420p -crf 23 /content/trace_h264.mp4

In [ ]:
import cv2, numpy as np, pandas as pd
from IPython.display import Image, display
d=pd.read_parquet('/content/clip90/tracks_masked.parquet')
C={"car":(0,200,255),"LGV":(0,255,120),"HGV":(255,80,0),"bus":(255,0,200),
   "motorcycle":(80,80,255),"cyclist":(255,255,0),"pedestrian":(0,255,255),
   "three-wheeler":(180,120,255)}
canvas=(ref*0.30).astype(np.uint8)
for tid,g in d.groupby('track_id'):
    g=g.sort_values('frame_idx')
    if np.hypot(g.cx.iloc[-1]-g.cx.iloc[0],g.cy_foot.iloc[-1]-g.cy_foot.iloc[0])<60: continue
    pts=np.stack([g.cx,g.cy_foot],1).astype(np.int32)
    cv2.polylines(canvas,[pts],False,C.get(g.class_group.iloc[0],(255,255,255)),2,cv2.LINE_AA)
cv2.imwrite('/content/all_traces.png',canvas)
display(Image('/content/all_traces.png',width=1400))

!mkdir -p /content/drive/MyDrive/flytbase_out
!cp /content/trace_h264.mp4 /content/all_traces.png /content/mask_check.png \
    /content/clip90/tracks_masked.parquet /content/clip90/track_quality.csv \
    /content/drive/MyDrive/flytbase_out/
!ls -la /content/drive/MyDrive/flytbase_out/

# level2

In [ ]:
import re, numpy as np
srt=open('/content/drive/MyDrive/flytbase_out/int.srt',errors='ignore').read()
g=lambda p: np.array([float(x) for x in re.findall(p,srt)])
alt=g(r'rel_alt:\s*([-\d.]+)'); pit=g(r'gb_pitch:\s*([-\d.]+)')
yaw=g(r'gb_yaw:\s*([-\d.]+)'); foc=g(r'focal_len:\s*([\d.]+)'); dz=g(r'dzoom_ratio:\s*([\d.]+)')
sl=slice(1798,4497)
print(f"n blocks {len(alt)}")
print(f"rel_alt {np.median(alt[sl]):.3f} (sd {alt[sl].std():.4f})")
print(f"pitch {np.median(pit[sl]):.2f}  yaw {np.median(yaw[sl]):.2f} (sd {yaw[sl].std():.3f})")
print(f"focal {np.median(foc[sl]):.2f}  dzoom {np.median(dz[sl]):.2f}")
REL_ALT=float(np.median(alt[sl])); PITCH=float(np.median(pit[sl])); YAW=float(np.median(yaw[sl]))

In [ ]:
import numpy as np
W_img,H_img=3840,2160
FOCAL_EQ=24.0
f_px=FOCAL_EQ/36.0*W_img
cx,cy=W_img/2,H_img/2
th=np.deg2rad(90+PITCH)          # tilt from nadir
c,s=np.cos(th),np.sin(th)
az=np.deg2rad(YAW)               # camera azimuth, clockwise from north

M=np.array([[REL_ALT/f_px, 0, -REL_ALT*cx/f_px],
            [0, -REL_ALT*c/f_px, REL_ALT*(cy*c/f_px+s)],
            [0, s/f_px, -cy*s/f_px+c]])

def to_ground(u,v):
    """image px -> local metres (X right-of-camera, Y away-from-camera)"""
    p=M@np.array([u,v,1.0]); return p[0]/p[2], p[1]/p[2]

def to_enu(u,v):
    """image px -> metres East/North"""
    X,Y=to_ground(u,v)
    return X*np.cos(az)+Y*np.sin(az), -X*np.sin(az)+Y*np.cos(az)

print(f"f_px {f_px:.0f}  tilt {np.degrees(th):.2f} deg")
for nm,v in [("top",0),("centre",1080),("bottom",2159)]:
    a=np.array(to_ground(1920,v)); b=np.array(to_ground(2020,v))
    d=np.array(to_ground(1920,v+10 if v<2100 else v-10))
    print(f"{nm:7s} across {np.linalg.norm(b-a):.3f} m/100px | along {np.linalg.norm(d-a)*10:.3f} m/100px")

In [ ]:
import pandas as pd, numpy as np
d=pd.read_parquet('/content/drive/MyDrive/flytbase_out/tracks_masked.parquet')

def box_dims(r):
    p=np.array([to_ground(r.x1,r.y1),to_ground(r.x2,r.y1),
                to_ground(r.x2,r.y2),to_ground(r.x1,r.y2)])
    return max(np.linalg.norm(p[1]-p[0]),np.linalg.norm(p[2]-p[1])), \
           min(np.linalg.norm(p[1]-p[0]),np.linalg.norm(p[2]-p[1]))

for cls,exp in [('car',4.2),('bus',11.0),('motorcycle',2.0)]:
    sub=d[d.class_group==cls]
    if len(sub)<20: continue
    sm=sub.sample(min(400,len(sub)),random_state=0)
    L=sm.apply(lambda r: box_dims(r)[0],axis=1)
    print(f"{cls:11s} median {L.median():5.2f} m  p25 {L.quantile(.25):.2f} p75 {L.quantile(.75):.2f}  (expect ~{exp})")

cars=d[d.class_group=='car'].sample(min(400,len(d[d.class_group=='car'])),random_state=0)
SCALE_FIX=4.2/cars.apply(lambda r: box_dims(r)[0],axis=1).median()
print(f"\nSCALE_FIX = {SCALE_FIX:.3f}")

In [ ]:
import numpy as np, pandas as pd
from scipy.signal import savgol_filter
SCALE_FIX = 1.0        # ← from L2-3
FPS,STRIDE=29.97,3; dt=STRIDE/FPS

d=pd.read_parquet('/content/drive/MyDrive/flytbase_out/tracks_masked.parquet')
E,N=zip(*[to_enu(x,y) for x,y in zip(d.cx,d.cy_foot)])
d['E']=np.array(E)*SCALE_FIX; d['N']=np.array(N)*SCALE_FIX

out=[]
for tid,t in d.groupby('track_id'):
    t=t.sort_values('frame_idx').copy(); n=len(t)
    if n<5: continue
    win=min(11,n if n%2 else n-1)
    if win<5: continue
    vx=savgol_filter(t.E.values,win,2,deriv=1,delta=dt)
    vy=savgol_filter(t.N.values,win,2,deriv=1,delta=dt)
    ax=savgol_filter(t.E.values,win,2,deriv=2,delta=dt)
    ay=savgol_filter(t.N.values,win,2,deriv=2,delta=dt)
    sp=np.hypot(vx,vy)
    with np.errstate(invalid='ignore',divide='ignore'):
        at=(ax*vx+ay*vy)/np.where(sp>0.1,sp,np.nan)
    t['E_s']=savgol_filter(t.E.values,win,2); t['N_s']=savgol_filter(t.N.values,win,2)
    t['speed_mps']=sp; t['speed_kmh']=sp*3.6
    t['accel_mps2']=np.nan_to_num(at)
    t['heading_deg']=np.degrees(np.arctan2(vx,vy))%360
    out.append(t)
k=pd.concat(out,ignore_index=True)
k.to_parquet('/content/kinematics.parquet',index=False)
print(k.groupby('class_group').speed_kmh.describe()[['count','25%','50%','75%','max']].round(1))
print("\naccel p1/p99:",k.accel_mps2.quantile([.01,.99]).round(2).tolist())

In [ ]:
import numpy as np, pandas as pd
k=pd.read_parquet('/content/kinematics.parquet')
md=[]
for tid,t in k.groupby('track_id'):
    mv=t[t.speed_mps>1.0]
    if len(mv)<5: mv=t
    dims=[box_dims(r) for r in mv.itertuples()]
    L=np.median([a for a,_ in dims])*SCALE_FIX
    Wd=np.median([b for _,b in dims])*SCALE_FIX
    md.append(dict(track_id=tid,L=L,Wd=Wd,det_cls=t.class_group.iloc[0],
                   v85=np.percentile(t.speed_kmh,85)))
m=pd.DataFrame(md)

def refine(r):
    c,L=r.det_cls,r.L
    if c in ('pedestrian','cyclist','motorcycle','three-wheeler'): return c
    if c=='bus': return 'bus'
    if L<5.5: return 'car'
    if L<7.5: return 'LGV'
    if L<10.5: return 'HGV-rigid'
    return 'HGV-artic'
m['class_final']=m.apply(refine,axis=1)
m.to_csv('/content/class_refined.csv',index=False)
k=k.merge(m[['track_id','L','Wd','class_final']],on='track_id',how='left')
k.to_parquet('/content/kinematics.parquet',index=False)
print(pd.crosstab(m.det_cls,m.class_final))
print("\nmedian length by final class:"); print(m.groupby('class_final').L.median().round(2))

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, pandas as pd, numpy as np
k=pd.read_parquet('/content/kinematics.parquet')
fig,ax=plt.subplots(1,3,figsize=(19,5))
for c,g in k.groupby('class_final'):
    if len(g)<200: continue
    ax[0].hist(g.speed_kmh,bins=60,histtype='step',density=True,label=c,lw=1.6)
ax[0].set_xlim(0,90); ax[0].set_xlabel('km/h'); ax[0].legend(fontsize=7)
ax[0].set_title('Speed distribution by class')
ax[1].hist(k.accel_mps2.clip(-4,4),bins=80,color='steelblue')
ax[1].set_xlabel('tangential accel (m/s²)'); ax[1].set_title('Acceleration')
for tid in k.groupby('track_id').size().sort_values(ascending=False).head(8).index:
    t=k[k.track_id==tid].sort_values('time_sec')
    ax[2].plot(t.time_sec-t.time_sec.min(),t.speed_kmh,lw=1.3)
ax[2].set_xlabel('t (s)'); ax[2].set_ylabel('km/h'); ax[2].set_title('Individual speed profiles')
plt.tight_layout(); plt.savefig('/content/l2_kinematics.png',dpi=130)

k[['frame_idx','time_sec','track_id','class_final','E_s','N_s','speed_mps','speed_kmh',
   'accel_mps2','heading_deg','L','Wd','interpolated']]\
 .to_parquet('/content/drive/MyDrive/flytbase_out/level2_kinematics.parquet',index=False)
!cp /content/l2_kinematics.png /content/class_refined.csv /content/drive/MyDrive/flytbase_out/
from IPython.display import Image, display
display(Image('/content/l2_kinematics.png',width=1400))

In [ ]:
# ===== L2 VALIDATION — run after L2-1 (REL_ALT, PITCH, YAW set) =====
import re, numpy as np, pandas as pd
from scipy.signal import savgol_filter

# --- telemetry (re-read so this cell is self-contained) ---
srt=open('/content/drive/MyDrive/flytbase_out/int.srt',errors='ignore').read()
g=lambda p: np.array([float(x) for x in re.findall(p,srt)])
sl=slice(1798,4497)
REL_ALT=float(np.median(g(r'rel_alt:\s*([-\d.]+)')[sl]))
PITCH  =float(np.median(g(r'gb_pitch:\s*([-\d.]+)')[sl]))
YAW    =float(np.median(g(r'gb_yaw:\s*([-\d.]+)')[sl]))
FOCAL  =float(np.median(g(r'focal_len:\s*([\d.]+)')[sl]))
DZ     =float(np.median(g(r'dzoom_ratio:\s*([\d.]+)')[sl]))
print(f"[TELEM] alt={REL_ALT:.3f} pitch={PITCH:.2f} yaw={YAW:.2f} focal={FOCAL:.2f} dzoom={DZ:.2f}")

# --- homography ---
W_img,H_img=3840,2160
f_px=FOCAL/36.0*W_img*DZ
cx,cy=W_img/2,H_img/2
th=np.deg2rad(90+PITCH); c,s=np.cos(th),np.sin(th); az=np.deg2rad(YAW)
M=np.array([[REL_ALT/f_px,0,-REL_ALT*cx/f_px],
            [0,-REL_ALT*c/f_px,REL_ALT*(cy*c/f_px+s)],
            [0,s/f_px,-cy*s/f_px+c]])
def to_ground(u,v):
    p=M@np.array([u,v,1.0]); return p[0]/p[2],p[1]/p[2]
def to_enu(u,v):
    X,Y=to_ground(u,v); return X*np.cos(az)+Y*np.sin(az), -X*np.sin(az)+Y*np.cos(az)
print(f"[GEOM] f_px={f_px:.0f} tilt={np.degrees(th):.2f}deg")
print("[GSD]", {nm:round(np.linalg.norm(np.array(to_ground(2020,v))-np.array(to_ground(1920,v))),3)
                for nm,v in [("top",0),("mid",1080),("bot",2159)]}, "m per 100px across-track")
print("[EXTENT corners m]", [tuple(np.round(to_ground(u,v),1)) for u,v in
      [(0,0),(3839,0),(0,2159),(3839,2159)]])

# --- vehicle dimension check ---
d=pd.read_parquet('/content/drive/MyDrive/flytbase_out/tracks_masked.parquet')
def box_dims(r):
    p=np.array([to_ground(r.x1,r.y1),to_ground(r.x2,r.y1),
                to_ground(r.x2,r.y2),to_ground(r.x1,r.y2)])
    a=np.linalg.norm(p[1]-p[0]); b=np.linalg.norm(p[2]-p[1])
    return max(a,b),min(a,b)
print("\n[DIMENSIONS]")
for cls,exp in [('car',4.2),('bus',11.0),('motorcycle',2.0),('LGV',5.5),('HGV',8.0)]:
    sub=d[d.class_group==cls]
    if len(sub)<20: continue
    sm=sub.sample(min(400,len(sub)),random_state=0)
    L=sm.apply(lambda r: box_dims(r)[0],axis=1)
    print(f"  {cls:11s} n={len(sub):6d} median {L.median():5.2f} m  p25 {L.quantile(.25):5.2f} "
          f"p75 {L.quantile(.75):5.2f}   expect ~{exp}")

# --- kinematics ---
FPS,STRIDE=29.97,3; dt=STRIDE/FPS
E,N=zip(*[to_enu(x,y) for x,y in zip(d.cx,d.cy_foot)])
d['E'],d['N']=np.array(E),np.array(N)
out=[]
for tid,t in d.groupby('track_id'):
    t=t.sort_values('frame_idx').copy(); n=len(t)
    if n<5: continue
    win=min(11,n if n%2 else n-1)
    if win<5: continue
    vx=savgol_filter(t.E.values,win,2,deriv=1,delta=dt)
    vy=savgol_filter(t.N.values,win,2,deriv=1,delta=dt)
    ax=savgol_filter(t.E.values,win,2,deriv=2,delta=dt)
    ay=savgol_filter(t.N.values,win,2,deriv=2,delta=dt)
    sp=np.hypot(vx,vy)
    with np.errstate(invalid='ignore',divide='ignore'):
        at=(ax*vx+ay*vy)/np.where(sp>0.1,sp,np.nan)
    t['E_s']=savgol_filter(t.E.values,win,2); t['N_s']=savgol_filter(t.N.values,win,2)
    t['speed_mps']=sp; t['speed_kmh']=sp*3.6; t['accel_mps2']=np.nan_to_num(at)
    t['heading_deg']=np.degrees(np.arctan2(vx,vy))%360
    out.append(t)
k=pd.concat(out,ignore_index=True)
k.to_parquet('/content/kinematics.parquet',index=False)

print("\n[SPEED km/h by class]")
print(k.groupby('class_group').speed_kmh.describe()[['count','25%','50%','75%','max']].round(1))

p=k[k.class_group=='pedestrian']
if len(p):
    print(f"\n[PEDESTRIAN] median {p.speed_mps.median():.2f} m/s   "
          f"(walking = 1.2-1.5)   ratio to 1.35 = {1.35/p.speed_mps.median():.3f}")
print("[ACCEL p1/p99]", k.accel_mps2.quantile([.01,.99]).round(2).tolist())

print("\n[LONGEST TRACKS]")
for tid in k.groupby('track_id').size().sort_values(ascending=False).head(6).index:
    t=k[k.track_id==tid].sort_values('time_sec')
    dist=float(np.sum(np.hypot(np.diff(t.E_s),np.diff(t.N_s))))
    dur=float(t.time_sec.max()-t.time_sec.min())
    print(f"  id {tid:6d} {t.class_group.iloc[0]:11s} {dist:6.1f} m / {dur:5.1f} s "
          f"= {dist/max(dur,.1)*3.6:5.1f} km/h avg, peak {t.speed_kmh.max():5.1f}")

In [ ]:
import numpy as np, pandas as pd
k=pd.read_parquet('/content/kinematics.parquet')

# per-track net displacement — separates real traffic from static artefacts
disp={}
for tid,t in k.groupby('track_id'):
    t=t.sort_values('time_sec')
    disp[tid]=float(np.hypot(t.E_s.iloc[-1]-t.E_s.iloc[0], t.N_s.iloc[-1]-t.N_s.iloc[0]))
k['net_disp']=k.track_id.map(disp)

print("[TRACKS by net displacement]")
nd=pd.Series(disp)
print(f"  total {len(nd)}   <5m (static) {(nd<5).sum()}   >20m (real traffic) {(nd>20).sum()}")
print(nd.describe().round(1))

mov=k[k.net_disp>20]          # genuinely traversing road users
print(f"\n[MOVING ONLY] {mov.track_id.nunique()} tracks")
print(mov.groupby('class_group').speed_kmh.describe()[['count','25%','50%','75%','max']].round(1))

# pedestrian speed among pedestrians that actually walked somewhere
pw=mov[(mov.class_group=='pedestrian')&(mov.speed_mps>0.3)]
if len(pw):
    print(f"\n[WALKING PED] median {pw.speed_mps.median():.2f} m/s  p75 {pw.speed_mps.quantile(.75):.2f}"
          f"   -> scale ratio {1.35/pw.speed_mps.median():.3f}")

# vehicle length measured only on moving vehicles (box best aligned when in motion)
print("\n[DIMENSIONS, moving only]")
for cls in ['car','motorcycle','bus','LGV','HGV']:
    s=mov[(mov.class_group==cls)&(mov.speed_mps>3)]
    if len(s)<30: print(f"  {cls}: only {len(s)} moving rows"); continue
    L=s.sample(min(400,len(s)),random_state=0).apply(lambda r: box_dims(r)[0],axis=1)
    print(f"  {cls:11s} median {L.median():5.2f} m  p25 {L.quantile(.25):.2f} p75 {L.quantile(.75):.2f}")

# what the static tracks actually are
st=k[k.net_disp<5]
print(f"\n[STATIC TRACKS] {st.track_id.nunique()} tracks, {len(st)} rows "
      f"({100*len(st)/len(k):.0f}% of data)")
print(st.groupby('track_id').class_group.first().value_counts())

In [ ]:
import numpy as np, pandas as pd
d=pd.read_parquet('/content/drive/MyDrive/flytbase_out/tracks_masked.parquet')

def build(pitch, alt=70.469, focal=24.0):
    f=focal/36.0*3840; cx,cy=1920,1080
    th=np.deg2rad(90+pitch); c,s=np.cos(th),np.sin(th)
    M=np.array([[alt/f,0,-alt*cx/f],[0,-alt*c/f,alt*(cy*c/f+s)],[0,s/f,-cy*s/f+c]])
    def tg(u,v):
        p=M@np.array([u,v,1.0]); return p[0]/p[2],p[1]/p[2]
    return tg

FPS,ST=29.97,3; dt=ST/FPS
ped=d[d.class_group=='pedestrian']
best=None
for pitch in np.arange(-85,-35,1.0):
    tg=build(pitch)
    sp_near,sp_far=[],[]
    for tid,t in ped.groupby('track_id'):
        t=t.sort_values('frame_idx')
        if len(t)<8: continue
        g=np.array([tg(x,y) for x,y in zip(t.cx,t.cy_foot)])
        v=np.hypot(*np.diff(g,axis=0).T)/dt
        v=v[v>0.3]
        if not len(v): continue
        (sp_near if t.cy_foot.mean()>1400 else sp_far).append(np.median(v))
    if len(sp_near)<5 or len(sp_far)<5: continue
    n,f_=np.median(sp_near),np.median(sp_far)
    err=abs(np.log(n/f_))                      # want near == far
    if best is None or err<best[0]: best=(err,pitch,n,f_)
print(f"best pitch {best[1]:.0f}  near {best[2]:.2f} m/s  far {best[3]:.2f} m/s  (SRT says -63.1)")

In [ ]:
import numpy as np, pandas as pd
d=pd.read_parquet('/content/drive/MyDrive/flytbase_out/tracks_masked.parquet')
FPS,ST=29.97,3; dt=ST/FPS

def build(pitch, alt=70.469, focal=24.0):
    f=focal/36.0*3840; cx,cy=1920,1080
    th=np.deg2rad(90+pitch); c,s=np.cos(th),np.sin(th)
    M=np.array([[alt/f,0,-alt*cx/f],[0,-alt*c/f,alt*(cy*c/f+s)],[0,s/f,-cy*s/f+c]])
    return lambda u,v: (lambda p:(p[0]/p[2],p[1]/p[2]))(M@np.array([u,v,1.0]))

ped=d[d.class_group=='pedestrian']
def ped_speeds(tg):
    near,far=[],[]
    for tid,t in ped.groupby('track_id'):
        t=t.sort_values('frame_idx')
        if len(t)<8: continue
        g=np.array([tg(x,y) for x,y in zip(t.cx,t.cy_foot)])
        v=np.hypot(*np.diff(g,axis=0).T)/dt; v=v[v>0.3]
        if len(v): (near if t.cy_foot.mean()>1400 else far).append(np.median(v))
    return np.median(near),np.median(far),len(near),len(far)

best=None
for p in np.arange(-42,-30,0.25):
    n,f_,cn,cf=ped_speeds(build(p))
    if cn<5 or cf<5: continue
    e=abs(np.log(n/f_))
    if best is None or e<best[0]: best=(e,p,n,f_)
PITCH_FIT=best[1]
print(f"refined pitch {PITCH_FIT:.2f}   near {best[2]:.3f}  far {best[3]:.3f} m/s")

to_ground=build(PITCH_FIT)
YAW=-126.10; az=np.deg2rad(YAW)
def to_enu(u,v):
    X,Y=to_ground(u,v); return X*np.cos(az)+Y*np.sin(az), -X*np.sin(az)+Y*np.cos(az)

print("[EXTENT m]", [tuple(np.round(to_ground(u,v),1)) for u,v in
      [(0,0),(3839,0),(0,2159),(3839,2159)]])

def box_dims(r):
    p=np.array([to_ground(r.x1,r.y1),to_ground(r.x2,r.y1),
                to_ground(r.x2,r.y2),to_ground(r.x1,r.y2)])
    a=np.linalg.norm(p[1]-p[0]); b=np.linalg.norm(p[2]-p[1])
    return max(a,b),min(a,b)

# net displacement filter
E,N=zip(*[to_enu(x,y) for x,y in zip(d.cx,d.cy_foot)])
d['E'],d['N']=np.array(E),np.array(N)
nd=d.groupby('track_id').apply(lambda t:(lambda s:np.hypot(s.E.iloc[-1]-s.E.iloc[0],
     s.N.iloc[-1]-s.N.iloc[0]))(t.sort_values('frame_idx')))
d['net_disp']=d.track_id.map(nd)
mov=d[d.net_disp>20]

print("\n[DIMENSIONS moving, speed>3]")
for cls,exp in [('car',3.8),('motorcycle',1.9),('LGV',5.5),('HGV',8.0),('bus',10.5)]:
    s=mov[mov.class_group==cls]
    if len(s)<30: print(f"  {cls}: n={len(s)}"); continue
    L=s.sample(min(400,len(s)),random_state=0).apply(lambda r: box_dims(r)[0],axis=1)
    print(f"  {cls:11s} median {L.median():5.2f}  p25 {L.quantile(.25):.2f} p75 {L.quantile(.75):.2f}  expect ~{exp}")

In [ ]:
# ===== L2 FINAL — GCP-calibrated =====
import cv2, numpy as np, pandas as pd
from scipy.signal import savgol_filter

IMG=np.float32([[3400,0],[3780,310],[530,2060],[0,1780]])
Wm,Dm=17.9,130.91
GND=np.float32([[0,0],[Wm,0],[Wm,Dm],[0,Dm]])
H_i2g=cv2.getPerspectiveTransform(IMG,GND)
def to_ground(u,v):
    p=H_i2g@np.array([u,v,1.0]); return float(p[0]/p[2]),float(p[1]/p[2])
print("[GCP] quad reprojection:",[tuple(np.round(to_ground(*p),2)) for p in IMG])
for nm,(u,v) in [('far',(3590,155)),('mid',(1900,1030)),('near',(265,1920))]:
    a=np.array(to_ground(u,v)); b=np.array(to_ground(u+100,v))
    print(f"  {nm}: {np.linalg.norm(b-a):.2f} m per 100px across")

FPS,ST=29.97,3; dt=ST/FPS
d=pd.read_parquet('/content/drive/MyDrive/flytbase_out/tracks_masked.parquet')
XY=np.array([to_ground(x,y) for x,y in zip(d.cx,d.cy_foot)])
d['X'],d['Y']=XY[:,0],XY[:,1]

first=d.sort_values('frame_idx').groupby('track_id')[['X','Y']].first()
last =d.sort_values('frame_idx').groupby('track_id')[['X','Y']].last()
nd=np.hypot(last.X-first.X,last.Y-first.Y)
d['net_disp']=d.track_id.map(nd)

def box_dims(r):
    p=np.array([to_ground(r.x1,r.y1),to_ground(r.x2,r.y1),
                to_ground(r.x2,r.y2),to_ground(r.x1,r.y2)])
    a=np.linalg.norm(p[1]-p[0]); b=np.linalg.norm(p[2]-p[1])
    return max(a,b),min(a,b)

out=[]
for tid,t in d.groupby('track_id'):
    t=t.sort_values('frame_idx').copy(); n=len(t)
    if n<5: continue
    win=min(11,n if n%2 else n-1)
    if win<5: continue
    vx=savgol_filter(t.X.values,win,2,deriv=1,delta=dt)
    vy=savgol_filter(t.Y.values,win,2,deriv=1,delta=dt)
    ax=savgol_filter(t.X.values,win,2,deriv=2,delta=dt)
    ay=savgol_filter(t.Y.values,win,2,deriv=2,delta=dt)
    sp=np.hypot(vx,vy)
    with np.errstate(invalid='ignore',divide='ignore'):
        at=(ax*vx+ay*vy)/np.where(sp>0.1,sp,np.nan)
    t['X_s']=savgol_filter(t.X.values,win,2); t['Y_s']=savgol_filter(t.Y.values,win,2)
    t['speed_mps']=sp; t['speed_kmh']=sp*3.6
    t['accel_mps2']=np.nan_to_num(at)
    t['heading_deg']=np.degrees(np.arctan2(vx,vy))%360
    out.append(t)
k=pd.concat(out,ignore_index=True)

# fine-grained class from metric length (moving frames only)
md=[]
for tid,t in k.groupby('track_id'):
    mv=t[t.speed_mps>2]
    if len(mv)<5: mv=t
    dims=[box_dims(r) for r in mv.itertuples()]
    md.append(dict(track_id=tid,L=float(np.median([a for a,_ in dims])),
                   Wd=float(np.median([b for _,b in dims])),det=t.class_group.iloc[0]))
m=pd.DataFrame(md)
def refine(r):
    c,L=r.det,r.L
    if c in ('pedestrian','cyclist','motorcycle','three-wheeler'): return c
    if L<5.0: return 'car'
    if L<7.5: return 'LGV'
    if L<10.5: return 'HGV-rigid'
    return 'bus' if c=='bus' else 'HGV-artic'
m['class_final']=m.apply(refine,axis=1)
k=k.merge(m[['track_id','L','Wd','class_final']],on='track_id',how='left')
k.to_parquet('/content/kinematics.parquet',index=False)
m.to_csv('/content/class_refined.csv',index=False)

mov=k[k.net_disp>20]
print(f"\n[MOVING] {mov.track_id.nunique()} of {k.track_id.nunique()} tracks")
print(mov.groupby('class_final').speed_kmh.describe()[['count','25%','50%','75%','max']].round(1))
print("\n[LENGTH m by final class]")
print(m.groupby('class_final').L.median().round(2))
pw=mov[(mov.class_final=='pedestrian')&(mov.speed_mps>0.4)]
if len(pw): print(f"\n[WALKING PED] {pw.speed_mps.median():.2f} m/s (expect 1.2-1.5)")
print("[ACCEL p1/p99]",k.accel_mps2.quantile([.01,.99]).round(2).tolist())
print("\n[SAMPLE TRAVERSALS]")
for tid in mov.groupby('track_id').net_disp.first().sort_values(ascending=False).head(6).index:
    t=k[k.track_id==tid].sort_values('time_sec')
    dur=t.time_sec.max()-t.time_sec.min()
    print(f"  {t.class_final.iloc[0]:11s} {t.net_disp.iloc[0]:6.1f} m / {dur:5.1f} s "
          f"= {t.net_disp.iloc[0]/dur*3.6:5.1f} km/h avg")

In [ ]:
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt, pandas as pd, numpy as np
k=pd.read_parquet('/content/kinematics.parquet')
mov=k[k.net_disp>20]

fig,ax=plt.subplots(2,2,figsize=(16,10))
for c,g in mov.groupby('class_final'):
    if g.track_id.nunique()<5: continue
    ax[0,0].hist(g.speed_kmh,bins=60,histtype='step',density=True,label=c,lw=1.7)
ax[0,0].set_xlim(0,70); ax[0,0].set_xlabel('speed (km/h)'); ax[0,0].legend(fontsize=8)
ax[0,0].set_title('Speed distribution by class (moving road users)')

ax[0,1].hist(k.accel_mps2.clip(-5,5),bins=90,color='steelblue')
ax[0,1].set_xlabel('tangential acceleration (m/s²)'); ax[0,1].set_title('Acceleration')

for tid in mov.groupby('track_id').net_disp.first().sort_values(ascending=False).head(10).index:
    t=k[k.track_id==tid].sort_values('time_sec')
    ax[1,0].plot(t.time_sec-t.time_sec.min(),t.speed_kmh,lw=1.3)
ax[1,0].set_xlabel('time since entry (s)'); ax[1,0].set_ylabel('km/h')
ax[1,0].set_title('Individual speed profiles, longest traversals')

L=k.groupby('track_id').agg(L=('L','first'),c=('class_final','first'))
for c,g in L.groupby('c'):
    if len(g)<4: continue
    ax[1,1].scatter([c]*len(g),g.L,s=14,alpha=.5)
ax[1,1].set_ylabel('measured length (m)'); ax[1,1].set_title('Metric length by class')
ax[1,1].tick_params(axis='x',rotation=30)
plt.tight_layout(); plt.savefig('/content/l2_kinematics.png',dpi=130)

k[['frame_idx','time_sec','track_id','class_final','X_s','Y_s','speed_mps','speed_kmh',
   'accel_mps2','heading_deg','L','Wd','net_disp','interpolated']]\
 .to_parquet('/content/drive/MyDrive/flytbase_out/level2_kinematics.parquet',index=False)
!cp /content/l2_kinematics.png /content/class_refined.csv /content/drive/MyDrive/flytbase_out/
from IPython.display import Image, display
display(Image('/content/l2_kinematics.png',width=1400))

In [ ]:
# ===== L2 COMPLETE — run standalone, writes everything to Drive =====
import cv2, numpy as np, pandas as pd, os, json
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

OUT='/content/drive/MyDrive/flytbase_out'
os.makedirs(OUT,exist_ok=True)

# --- GCP homography (your 4 points, your Maps measurements) ---
IMG=np.float32([[3400,0],[3780,310],[530,2060],[0,1780]])
Wm,Dm=17.9,130.91
H_i2g=cv2.getPerspectiveTransform(IMG,np.float32([[0,0],[Wm,0],[Wm,Dm],[0,Dm]]))
def to_ground(u,v):
    p=H_i2g@np.array([u,v,1.0]); return float(p[0]/p[2]),float(p[1]/p[2])

FPS,ST=29.97,3; dt=ST/FPS
d=pd.read_parquet(f'{OUT}/tracks_masked.parquet')
XY=np.array([to_ground(x,y) for x,y in zip(d.cx,d.cy_foot)])
d['X'],d['Y']=XY[:,0],XY[:,1]
f=d.sort_values('frame_idx').groupby('track_id')[['X','Y']].first()
l=d.sort_values('frame_idx').groupby('track_id')[['X','Y']].last()
d['net_disp']=d.track_id.map(np.hypot(l.X-f.X,l.Y-f.Y))

def box_dims(r):
    p=np.array([to_ground(r.x1,r.y1),to_ground(r.x2,r.y1),
                to_ground(r.x2,r.y2),to_ground(r.x1,r.y2)])
    a=np.linalg.norm(p[1]-p[0]); b=np.linalg.norm(p[2]-p[1])
    return max(a,b),min(a,b)

out=[]
for tid,t in d.groupby('track_id'):
    t=t.sort_values('frame_idx').copy(); n=len(t)
    if n<5: continue
    win=min(15,n if n%2 else n-1)
    if win<5: continue
    vx=savgol_filter(t.X.values,win,2,deriv=1,delta=dt)
    vy=savgol_filter(t.Y.values,win,2,deriv=1,delta=dt)
    ax=savgol_filter(t.X.values,win,2,deriv=2,delta=dt)
    ay=savgol_filter(t.Y.values,win,2,deriv=2,delta=dt)
    sp=np.hypot(vx,vy)
    with np.errstate(invalid='ignore',divide='ignore'):
        at=(ax*vx+ay*vy)/np.where(sp>0.1,sp,np.nan)
    t['X_s']=savgol_filter(t.X.values,win,2); t['Y_s']=savgol_filter(t.Y.values,win,2)
    t['speed_mps']=sp; t['speed_kmh']=sp*3.6
    t['accel_mps2']=np.nan_to_num(at)
    t['heading_deg']=np.degrees(np.arctan2(vx,vy))%360
    out.append(t)
k=pd.concat(out,ignore_index=True)

md=[]
for tid,t in k.groupby('track_id'):
    mv=t[t.speed_mps>2]
    if len(mv)<5: mv=t
    dims=[box_dims(r) for r in mv.itertuples()]
    md.append(dict(track_id=tid,L=float(np.median([a for a,_ in dims])),
                   Wd=float(np.median([b for _,b in dims])),det_cls=t.class_group.iloc[0],
                   net_disp=float(t.net_disp.iloc[0]),
                   dur_s=float(t.time_sec.max()-t.time_sec.min()),
                   v_med=float(t.speed_kmh.median()),v85=float(np.percentile(t.speed_kmh,85))))
m=pd.DataFrame(md)
def refine(r):
    c,L=r.det_cls,r.L
    if c in ('pedestrian','cyclist','motorcycle','three-wheeler'): return c
    if L<5.0: return 'car'
    if L<7.5: return 'LGV'
    if L<10.5: return 'HGV-rigid'
    return 'bus' if c=='bus' else 'HGV-artic'
m['class_final']=m.apply(refine,axis=1)
k=k.merge(m[['track_id','L','Wd','class_final']],on='track_id',how='left')
mov=k[k.net_disp>20]

# ---------- plots ----------
fig,ax=plt.subplots(2,2,figsize=(16,10))
for c,g in mov.groupby('class_final'):
    if g.track_id.nunique()<5: continue
    ax[0,0].hist(g.speed_kmh,bins=60,histtype='step',density=True,label=c,lw=1.7)
ax[0,0].set_xlim(0,70); ax[0,0].set_xlabel('speed (km/h)'); ax[0,0].legend(fontsize=8)
ax[0,0].set_title('Speed distribution by class (moving road users)')
ax[0,1].hist(k.accel_mps2.clip(-5,5),bins=90,color='steelblue')
ax[0,1].set_xlabel('tangential acceleration (m/s²)'); ax[0,1].set_title('Acceleration')
for tid in mov.groupby('track_id').net_disp.first().sort_values(ascending=False).head(10).index:
    t=k[k.track_id==tid].sort_values('time_sec')
    ax[1,0].plot(t.time_sec-t.time_sec.min(),t.speed_kmh,lw=1.3)
ax[1,0].set_xlabel('time since entry (s)'); ax[1,0].set_ylabel('km/h')
ax[1,0].set_title('Speed profiles, longest traversals')
for c,g in m.groupby('class_final'):
    if len(g)<4: continue
    ax[1,1].scatter([c]*len(g),g.L,s=14,alpha=.5)
ax[1,1].set_ylabel('measured length (m)'); ax[1,1].set_title('Metric length by class')
ax[1,1].tick_params(axis='x',rotation=30)
plt.tight_layout(); plt.savefig(f'{OUT}/l2_kinematics.png',dpi=130)

# ---------- exports ----------
k[['frame_idx','time_sec','track_id','class_final','X_s','Y_s','speed_mps','speed_kmh',
   'accel_mps2','heading_deg','L','Wd','net_disp','interpolated']]\
 .to_parquet(f'{OUT}/level2_kinematics.parquet',index=False)
m.to_csv(f'{OUT}/level2_per_track.csv',index=False)

tbl=mov.groupby('class_final').speed_kmh.describe()[['count','25%','50%','75%','max']].round(1)
tbl['tracks']=mov.groupby('class_final').track_id.nunique()
tbl['median_len_m']=m.groupby('class_final').L.median().round(2)
tbl.to_csv(f'{OUT}/level2_summary.csv')

pw=mov[(mov.class_final=='pedestrian')&(mov.speed_mps>0.4)]
gsd={nm:round(np.linalg.norm(np.array(to_ground(u+100,v))-np.array(to_ground(u,v))),3)
     for nm,(u,v) in [('far',(3590,155)),('mid',(1900,1030)),('near',(265,1920))]}
val=dict(gcp_points=IMG.tolist(),corridor_width_m=Wm,corridor_length_m=Dm,
         gsd_m_per_100px=gsd,
         ped_walk_mps=round(float(pw.speed_mps.median()),3) if len(pw) else None,
         accel_p1_p99=[round(x,2) for x in k.accel_mps2.quantile([.01,.99])],
         tracks_total=int(k.track_id.nunique()),tracks_moving=int(mov.track_id.nunique()),
         median_length_m=m.groupby('class_final').L.median().round(2).to_dict())
json.dump(val,open(f'{OUT}/level2_validation.json','w'),indent=2)

print("[GSD m per 100px]",gsd)
print(f"[WALKING PED] {val['ped_walk_mps']} m/s (expect 1.2-1.5)")
print(f"[ACCEL p1/p99] {val['accel_p1_p99']}")
print(f"[TRACKS] {val['tracks_moving']} moving of {val['tracks_total']}\n")
print(tbl)
print("\n[LENGTH m]"); print(m.groupby('class_final').L.median().round(2))
print("\n[TRAVERSALS]")
for tid in mov.groupby('track_id').net_disp.first().sort_values(ascending=False).head(6).index:
    t=k[k.track_id==tid].sort_values('time_sec'); du=t.time_sec.max()-t.time_sec.min()
    print(f"  {t.class_final.iloc[0]:12s} {t.net_disp.iloc[0]:6.1f} m / {du:5.1f} s = {t.net_disp.iloc[0]/du*3.6:5.1f} km/h")
print("\nWROTE:"); [print("  ",x) for x in sorted(os.listdir(OUT))]

In [ ]:
import pandas as pd, numpy as np, os
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

OUT='/content/drive/MyDrive/flytbase_out'
k=pd.read_parquet(f'{OUT}/level2_kinematics.parquet')
m=pd.read_csv(f'{OUT}/level2_per_track.csv')
mov=k[k.net_disp>20]

g=mov.groupby('class_final')
t=pd.DataFrame({
    'Tracks': g.track_id.nunique(),
    'Median km/h': g.speed_kmh.median().round(1),
    'IQR km/h': g.speed_kmh.quantile(.25).round(1).astype(str)+'–'+g.speed_kmh.quantile(.75).round(1).astype(str),
    'Max km/h': g.speed_kmh.max().round(1),
    'Length m': m.groupby('class_final').L.median().round(2),
})
t=t.sort_values('Tracks',ascending=False)
t.index.name='Class'
t=t.reset_index()

# ---- PNG ----
fig,ax=plt.subplots(figsize=(9,0.55*len(t)+1.3)); ax.axis('off')
tb=ax.table(cellText=t.values,colLabels=t.columns,cellLoc='center',loc='center')
tb.auto_set_font_size(False); tb.set_fontsize(11); tb.scale(1,1.6)
for j in range(len(t.columns)):
    c=tb[0,j]; c.set_facecolor('#1f3a5f'); c.set_text_props(color='white',weight='bold')
for i in range(1,len(t)+1):
    for j in range(len(t.columns)):
        tb[i,j].set_facecolor('#f2f5f9' if i%2 else 'white')
        if t.iloc[i-1]['Tracks']<5: tb[i,j].set_text_props(color='#a04000')
ax.set_title('Level 2 — Speed and size by class (90 s window, 167 moving road users)',
             fontsize=12,weight='bold',pad=16)
plt.tight_layout(); plt.savefig(f'{OUT}/level2_table.png',dpi=200,bbox_inches='tight')

# ---- markdown ----
md=t.to_markdown(index=False)
open(f'{OUT}/level2_table.md','w').write(
 "## Observed traffic — 90 s window, 167 moving road users of 316 total\n\n"+md+
 "\n\nRows with fewer than 5 tracks are single observations, not class distributions.\n")
print(md)

from IPython.display import Image, display
display(Image(f'{OUT}/level2_table.png'))

In [ ]:
%%writefile label.py
import sys, cv2, numpy as np, pandas as pd
C={"car":(0,200,255),"LGV":(0,255,120),"HGV-rigid":(255,80,0),"HGV-artic":(255,120,0),
   "bus":(255,0,200),"motorcycle":(80,80,255),"cyclist":(255,255,0),
   "pedestrian":(0,255,255),"three-wheeler":(180,120,255)}
kin,box,vid,mk,out = sys.argv[1:5+1][:5]
TAIL=90
k=pd.read_parquet(kin)                      # has speed/class
b=pd.read_parquet(box)                      # has x1..y2
d=b.merge(k[['frame_idx','track_id','speed_kmh','class_final','L','net_disp']],
          on=['frame_idx','track_id'],how='inner')
mask=np.load(mk)
st=int(np.median(np.diff(sorted(d.frame_idx.unique())))) or 3
cap=cv2.VideoCapture(vid); assert cap.isOpened()
fps=cap.get(cv2.CAP_PROP_FPS) or 29.97
W=int(cap.get(3)); H=int(cap.get(4)); OW=1920; OH=int(1920*H/W); sx,sy=OW/W,OH/H
msk=cv2.resize(mask,(OW,OH))
f0,f1=int(d.frame_idx.min()),int(d.frame_idx.max())
by={kk:v for kk,v in d.groupby("frame_idx")}; hist={}
vw=cv2.VideoWriter(out,cv2.VideoWriter_fourcc(*"mp4v"),fps/st,(OW,OH))
cap.set(cv2.CAP_PROP_POS_FRAMES,f0); i=f0; n=0
while i<=f1:
    if not cap.grab(): break
    if i in by:
        ok,fr=cap.retrieve()
        if not ok: i+=1; continue
        fr=cv2.resize(fr,(OW,OH))
        fr[msk==0]=(fr[msk==0]*0.4).astype(np.uint8)
        layer=fr.copy()
        rows=by[i]
        for r in rows.itertuples():
            c=C.get(r.class_final,(255,255,255))
            p=(int(r.cx*sx),int(r.cy_foot*sy))
            hist.setdefault(r.track_id,[]).append(p)
            pts=hist[r.track_id][-TAIL:]
            for q in range(1,len(pts)):
                a=q/len(pts)
                cv2.line(layer,pts[q-1],pts[q],tuple(int(v*a) for v in c),
                         max(1,int(1+2*a)),cv2.LINE_AA)
        fr=cv2.addWeighted(layer,0.8,fr,0.2,0)
        for r in rows.itertuples():
            c=C.get(r.class_final,(255,255,255))
            x1,y1=int(r.x1*sx),int(r.y1*sy); x2,y2=int(r.x2*sx),int(r.y2*sy)
            cv2.rectangle(fr,(x1,y1),(x2,y2),c,2)
            lab=f"{r.class_final} {r.speed_kmh:.0f}km/h"
            (tw,th),_=cv2.getTextSize(lab,cv2.FONT_HERSHEY_SIMPLEX,0.42,1)
            ty=max(y1-6,th+4)
            cv2.rectangle(fr,(x1,ty-th-4),(x1+tw+6,ty+3),(0,0,0),-1)
            cv2.putText(fr,lab,(x1+3,ty),cv2.FONT_HERSHEY_SIMPLEX,0.42,c,1,cv2.LINE_AA)
            cv2.putText(fr,f"#{int(r.track_id)} {r.L:.1f}m",(x1+3,y2+13),
                        cv2.FONT_HERSHEY_SIMPLEX,0.36,c,1,cv2.LINE_AA)
        mvn=rows[rows.net_disp>20]
        cv2.rectangle(fr,(0,0),(560,74),(0,0,0),-1)
        cv2.putText(fr,f"t={i/fps:6.1f}s   active={len(rows)}   moving={len(mvn)}",
                    (12,28),cv2.FONT_HERSHEY_SIMPLEX,0.72,(255,255,255),2,cv2.LINE_AA)
        cv2.putText(fr,f"mean speed {mvn.speed_kmh.mean():.1f} km/h" if len(mvn) else "",
                    (12,58),cv2.FONT_HERSHEY_SIMPLEX,0.62,(180,220,255),1,cv2.LINE_AA)
        y=100
        for cls,col in C.items():
            if cls in set(d.class_final):
                cv2.rectangle(fr,(12,y-11),(30,y+3),col,-1)
                cv2.putText(fr,cls,(38,y),cv2.FONT_HERSHEY_SIMPLEX,0.5,(255,255,255),1,cv2.LINE_AA)
                y+=24
        vw.write(fr); n+=1
    i+=1
cap.release(); vw.release(); print(f"wrote {out}: {n} frames")

In [ ]:
OUT='/content/drive/MyDrive/flytbase_out'
!python label.py {OUT}/level2_kinematics.parquet {OUT}/tracks_masked.parquet \
  "$VID" /content/road_mask.npy /content/labeled.mp4
!ffmpeg -y -loglevel error -i /content/labeled.mp4 -vcodec libx264 -pix_fmt yuv420p -crf 23 \
  {OUT}/level2_labeled_h264.mp4
!ls -la {OUT}/level2_labeled_h264.mp4